# Notebook 123: LSTM ― 記憶を制御する
## Long Short-Term Memory: Gating the Information Flow
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の中核。NB122で実証した勾配消失問題に対する「外科的解決策」としてLSTMを学びます。

### 学習目標
1. **LSTMの4ゲート** （忘却・入力・セル候補・出力）の役割を説明できる
2. **セル状態の勾配高速道路** メカニズムを理解する
3. **LSTMCellのforward/backward** をNumPyで完全に実装できる
4. **Adding Problem** でRNN失敗 → LSTM成功を実証できる
5. **ゲート活性化の可視化** から情報フローを読み解ける

### 前提知識
- Notebook 121（バニラRNN）、Notebook 122（BPTTと勾配消失）

⏱️ **推定学習時間**: 150-180分  
📊 **難易度**: ★★★★☆（上級）  
🎓 **カテゴリ**: 理論・実装

## 目次

1. [なぜLSTMが必要か](#1.-なぜLSTMが必要か)
2. [LSTMの構造 ― 4つのゲート](#2.-LSTMの構造-―-4つのゲート)
3. [セル状態: 勾配の高速道路](#3.-セル状態:-勾配の高速道路)
4. [LSTMCell Forward実装](#4.-LSTMCell-Forward実装)
5. [LSTMCell Backward実装](#5.-LSTMCell-Backward実装)
6. [勾配チェック](#6.-勾配チェック)
7. [Adding Problemでの実証](#7.-Adding-Problemでの実証)
8. [ゲート活性化の可視化](#8.-ゲート活性化の可視化)
9. [まとめ](#9.-まとめ)
10. [自己評価クイズ](#10.-自己評価クイズ)

In [ ]:
# ============================================================
# セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from copy import deepcopy

# 再現性の確保
np.random.seed(42)

# プロットスタイル
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# 日本語フォント設定（環境に応じて変更）
try:
    plt.rcParams['font.family'] = 'Hiragino Sans'
except:
    pass

print("NumPy version:", np.__version__)
print("セットアップ完了！")

## 1. なぜLSTMが必要か

### RNNの勾配消失問題（NB122の復習）

バニラRNNでは、時刻 $t$ から時刻 $k$ への勾配は以下の連鎖律で計算されます：

$$\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} = \prod_{j=k+1}^{t} \text{diag}(1 - h_j^2) \cdot W_{hh}$$

この積は $t - k$ が大きくなると：
- $\|W_{hh}\|$ の最大特異値 $< 1$ → **指数的に減衰**（勾配消失）
- $\|W_{hh}\|$ の最大特異値 $> 1$ → **指数的に爆発**（勾配爆発）

### 必要なもの

勾配が長距離を通過できる「バイパス」＝ **セル状態 (cell state)** という概念が必要です。

| 問題 | RNNの状況 | LSTMの解決策 |
|------|----------|-------------|
| 勾配消失 | $W_{hh}$ の行列積が指数的に縮小 | セル状態による加算的更新 |
| 長期記憶 | 遠い過去の情報が失われる | 忘却ゲートで選択的に保持 |
| 情報選択 | すべての情報が混合される | ゲートで情報フローを制御 |

## 2. LSTMの構造 ― 4つのゲート

LSTMの各時刻 $t$ での計算は以下の6つの式で表されます：

### ゲート方程式

| ゲート | 式 | 役割 |
|--------|-----|------|
| **忘却ゲート** | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | 前のセル状態をどれだけ忘れるか |
| **入力ゲート** | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | 新しい情報をどれだけ書き込むか |
| **セル候補** | $\tilde{c}_t = \tanh(W_c [h_{t-1}, x_t] + b_c)$ | 書き込む候補値 |
| **出力ゲート** | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | セル状態のどの部分を出力するか |

### セル状態と隠れ状態の更新

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

$$h_t = o_t \odot \tanh(c_t)$$

ここで $\odot$ はアダマール積（要素ごとの積）、$\sigma$ はシグモイド関数です。

### LSTM構造図（ASCII）

```
                    c_{t-1}                          c_t
              ─────────┬──────────────────────────────┬───────────
                       │           ┌──────┐           │
                       ├─── × ─────┤  f_t │           │
                       │           └──────┘           │
                       │                              │
                       └──────── + ──────────────────►│
                                  ▲                   │
                       ┌──────┐   │   ┌──────┐       │
                       │  i_t ├── × ──┤ g_t  │       │
                       └──────┘       └──────┘       │
                                                      │
              h_{t-1}                          ┌──────┤
              ─────────┬──────────────────────►│ tanh │
                       │                       └──┬───┘
                       │           ┌──────┐       │
                       ├───────────┤  o_t ├── × ──┴──► h_t
                       │           └──────┘
                       │
              x_t ─────┘
```

**ポイント**: セル状態 $c_t$ は上部を「高速道路」のように流れ、ゲートで選択的に読み書きされます。

In [ ]:
# ============================================================
# ゲート関数の可視化: シグモイドとtanhの役割
# ============================================================

x = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# シグモイド関数
ax = axes[0]
ax.plot(x, sigmoid, 'b-', linewidth=2.5, label=r'$\sigma(x)$')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axhline(y=1, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(x, sigmoid, alpha=0.1, color='blue')
ax.set_title('シグモイド関数 σ(x)\n（ゲート: 0=閉, 1=開）', fontsize=14)
ax.set_xlabel('x')
ax.set_ylabel('σ(x)')
ax.set_ylim(-0.1, 1.1)
ax.annotate('閉じている\n(情報を遮断)', xy=(-4, 0.02), fontsize=11,
            color='red', ha='center')
ax.annotate('開いている\n(情報を通過)', xy=(4, 0.98), fontsize=11,
            color='green', ha='center')
ax.legend(fontsize=13)

# tanh関数
ax = axes[1]
ax.plot(x, tanh, 'r-', linewidth=2.5, label=r'$\tanh(x)$')
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)
ax.fill_between(x, tanh, alpha=0.1, color='red')
ax.set_title('tanh関数\n（値の範囲: -1〜+1）', fontsize=14)
ax.set_xlabel('x')
ax.set_ylabel('tanh(x)')
ax.set_ylim(-1.3, 1.3)
ax.annotate('負の情報', xy=(-4, -0.95), fontsize=11,
            color='blue', ha='center')
ax.annotate('正の情報', xy=(4, 0.95), fontsize=11,
            color='red', ha='center')
ax.legend(fontsize=13)

plt.suptitle('LSTMゲートで使われる活性化関数', fontsize=16, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

print("σ(x): 忘却ゲート(f)、入力ゲート(i)、出力ゲート(o) で使用 → 0〜1の値でゲート開閉を制御")
print("tanh(x): セル候補(g)、セル状態の出力変換 で使用 → -1〜+1の値で情報を表現")

## 3. セル状態: 勾配の高速道路

### LSTMが勾配消失を解決する核心的メカニズム

セル状態の更新式を再確認します：

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

この式でのセル状態に関する勾配は：

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t \quad \text{（要素ごとの掛け算のみ！）}$$

### RNNとの決定的な違い

| | RNN | LSTM |
|---|-----|------|
| 勾配の伝搬 | $\frac{\partial h_t}{\partial h_{t-1}} = \text{diag}(1 - h_t^2) \cdot W_{hh}$ | $\frac{\partial c_t}{\partial c_{t-1}} = f_t$ |
| 演算 | **行列積** | **要素ごとの積** |
| $T$ ステップ後 | $\prod W_{hh}$ → 特異値の$T$乗 | $\prod f_t$ → $f_t \approx 1$ なら $\approx 1$ |
| 勾配の運命 | 消失 or 爆発 | **ゲートが制御** |

### 高速道路の比喩

- **RNN**: 信号が各交差点で複雑な変換（行列積）を受ける一般道
- **LSTM**: セル状態は高速道路 ― ゲート（IC/SA）でのみ乗り降りする
- 忘却ゲート $f_t \approx 1$ の場合、勾配は**ほぼ減衰なし**で長距離を伝搬可能

In [ ]:
# ============================================================
# 勾配フロー比較: RNN vs LSTM
# ============================================================

np.random.seed(42)

T = 50  # シーケンス長
hidden_dim = 32

# --- RNNの勾配ノルム ---
# W_hh の特異値が1より少し小さい場合をシミュレーション
W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.5
# 特異値を0.95に正規化
U, S, Vt = np.linalg.svd(W_hh)
S = S / S.max() * 0.95
W_hh = U @ np.diag(S) @ Vt

rnn_grad_norms = [1.0]
grad = np.eye(hidden_dim)
for t in range(T - 1):
    # diag(1 - h_t^2) を簡略化して0.8倍とする
    tanh_deriv = np.diag(np.random.uniform(0.5, 0.9, hidden_dim))
    grad = tanh_deriv @ W_hh @ grad
    rnn_grad_norms.append(np.linalg.norm(grad))

# --- LSTMの勾配ノルム ---
# 忘却ゲートが0.9〜1.0の値を取る場合
lstm_grad_norms = [1.0]
grad_cell = np.ones(hidden_dim)
for t in range(T - 1):
    f_t = np.random.uniform(0.9, 1.0, hidden_dim)  # 忘却ゲート ≈ 1
    grad_cell = grad_cell * f_t
    lstm_grad_norms.append(np.linalg.norm(grad_cell))

# --- 可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 線形スケール
ax = axes[0]
ax.plot(range(T), rnn_grad_norms, 'r-', linewidth=2, label='RNN', alpha=0.8)
ax.plot(range(T), lstm_grad_norms, 'b-', linewidth=2, label='LSTM', alpha=0.8)
ax.set_xlabel('時間ステップ（逆方向）', fontsize=12)
ax.set_ylabel('勾配ノルム', fontsize=12)
ax.set_title('勾配の減衰（線形スケール）', fontsize=14)
ax.legend(fontsize=12)

# 対数スケール
ax = axes[1]
ax.semilogy(range(T), rnn_grad_norms, 'r-', linewidth=2, label='RNN', alpha=0.8)
ax.semilogy(range(T), lstm_grad_norms, 'b-', linewidth=2, label='LSTM', alpha=0.8)
ax.set_xlabel('時間ステップ（逆方向）', fontsize=12)
ax.set_ylabel('勾配ノルム（対数スケール）', fontsize=12)
ax.set_title('勾配の減衰（対数スケール）', fontsize=14)
ax.legend(fontsize=12)

plt.suptitle('RNN vs LSTM: 勾配伝搬の比較', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"50ステップ後の勾配ノルム:")
print(f"  RNN:  {rnn_grad_norms[-1]:.6e}  (ほぼゼロ → 勾配消失)")
print(f"  LSTM: {lstm_grad_norms[-1]:.4f}  (維持されている → 長期依存性を学習可能)")

## 4. LSTMCell Forward実装

効率のため、4つのゲートの重みを1つの大きな行列に結合します：

$$\begin{bmatrix} i \\ f \\ g \\ o \end{bmatrix} = \begin{bmatrix} \sigma \\ \sigma \\ \tanh \\ \sigma \end{bmatrix} \left( W \begin{bmatrix} h_{t-1} \\ x_t \end{bmatrix} + b \right)$$

**重要なトリック**: 忘却ゲートのバイアスを1.0で初期化
- 訓練初期に $f_t \approx \sigma(1) \approx 0.73$ となり、セル状態が保持されやすくなる
- これなしでは $f_t \approx \sigma(0) = 0.5$ で情報が急速に失われる

In [ ]:
# ============================================================
# LSTMCell: Forward実装
# ============================================================

class LSTMCell:
    """NumPyによるLSTMセルの完全実装
    
    4つのゲートを1つの重み行列に結合して効率的に計算する。
    ゲートの順序: [input, forget, cell_candidate, output]
    """
    
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        
        # Xavier初期化
        scale = np.sqrt(2.0 / concat_dim)
        self.W = np.random.randn(concat_dim, 4 * hidden_dim) * scale
        self.b = np.zeros(4 * hidden_dim)
        
        # 忘却ゲートのバイアスを1.0で初期化（重要！）
        H = hidden_dim
        self.b[H:2*H] = 1.0
        
        # 勾配の初期化
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        
        # キャッシュ
        self.cache = None
    
    def _sigmoid(self, x):
        """数値安定なシグモイド関数"""
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, x_t, h_prev, c_prev):
        """LSTMセルの順伝搬
        
        Args:
            x_t:    入力 (batch_size, input_dim)
            h_prev: 前の隠れ状態 (batch_size, hidden_dim)
            c_prev: 前のセル状態 (batch_size, hidden_dim)
        
        Returns:
            h_next: 次の隠れ状態 (batch_size, hidden_dim)
            c_next: 次のセル状態 (batch_size, hidden_dim)
        """
        H = self.hidden_dim
        
        # 入力と前の隠れ状態を結合
        concat = np.concatenate([x_t, h_prev], axis=1)  # (batch, input+hidden)
        
        # 全ゲートを一括計算
        gates = concat @ self.W + self.b  # (batch, 4*hidden)
        
        # ゲートの分割と活性化
        i = self._sigmoid(gates[:, :H])         # 入力ゲート
        f = self._sigmoid(gates[:, H:2*H])      # 忘却ゲート
        g = np.tanh(gates[:, 2*H:3*H])          # セル候補
        o = self._sigmoid(gates[:, 3*H:])        # 出力ゲート
        
        # セル状態の更新
        c_next = f * c_prev + i * g
        
        # 隠れ状態の更新
        tanh_c_next = np.tanh(c_next)
        h_next = o * tanh_c_next
        
        # Backward用にキャッシュ
        self.cache = (concat, i, f, g, o, c_prev, c_next, tanh_c_next)
        
        return h_next, c_next


# --- 動作テスト ---
np.random.seed(42)
batch_size = 4
input_dim = 3
hidden_dim = 5

cell = LSTMCell(input_dim, hidden_dim)
x_t = np.random.randn(batch_size, input_dim)
h_prev = np.zeros((batch_size, hidden_dim))
c_prev = np.zeros((batch_size, hidden_dim))

h_next, c_next = cell.forward(x_t, h_prev, c_prev)

print("=== LSTMCell Forward テスト ===")
print(f"入力 x_t:     shape={x_t.shape}")
print(f"h_prev:       shape={h_prev.shape}")
print(f"c_prev:       shape={c_prev.shape}")
print(f"h_next:       shape={h_next.shape}, range=[{h_next.min():.4f}, {h_next.max():.4f}]")
print(f"c_next:       shape={c_next.shape}, range=[{c_next.min():.4f}, {c_next.max():.4f}]")
print()

# ゲート値の確認
_, i, f, g, o, _, _, _ = cell.cache
print("ゲート活性化（最初のサンプル）:")
print(f"  入力ゲート i: {i[0].round(3)}")
print(f"  忘却ゲート f: {f[0].round(3)}  ← バイアス=1.0により初期値が高い")
print(f"  セル候補   g: {g[0].round(3)}")
print(f"  出力ゲート o: {o[0].round(3)}")

## 5. LSTMCell Backward実装

### 逆伝搬の導出

順伝搬の各ステップを逆にたどって勾配を計算します。

上流からの勾配: $\delta h = \frac{\partial L}{\partial h_t}$, $\delta c_{\text{next}} = \frac{\partial L}{\partial c_t}$

1. **出力ゲートと$\tanh(c_t)$の勾配**:
   - $h_t = o_t \odot \tanh(c_t)$ より
   - $\delta o = \delta h \odot \tanh(c_t)$
   - $\delta c_t \mathrel{+}= \delta h \odot o_t \odot (1 - \tanh^2(c_t))$

2. **セル状態の勾配**:
   - $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ より
   - $\delta f = \delta c_t \odot c_{t-1}$
   - $\delta i = \delta c_t \odot \tilde{c}_t$
   - $\delta g = \delta c_t \odot i_t$
   - $\delta c_{t-1} = \delta c_t \odot f_t$ ← **加算的！これが核心**

3. **活性化関数の逆伝搬**:
   - シグモイドの導関数: $\sigma'(x) = \sigma(x)(1 - \sigma(x))$
   - tanhの導関数: $\tanh'(x) = 1 - \tanh^2(x)$

In [ ]:
# ============================================================
# LSTMCell: Backward実装を追加
# ============================================================

class LSTMCell:
    """NumPyによるLSTMセルの完全実装（forward + backward）"""
    
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        
        scale = np.sqrt(2.0 / concat_dim)
        self.W = np.random.randn(concat_dim, 4 * hidden_dim) * scale
        self.b = np.zeros(4 * hidden_dim)
        
        # 忘却ゲートのバイアスを1.0で初期化
        H = hidden_dim
        self.b[H:2*H] = 1.0
        
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.cache = None
    
    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, x_t, h_prev, c_prev):
        H = self.hidden_dim
        concat = np.concatenate([x_t, h_prev], axis=1)
        gates = concat @ self.W + self.b
        
        i = self._sigmoid(gates[:, :H])
        f = self._sigmoid(gates[:, H:2*H])
        g = np.tanh(gates[:, 2*H:3*H])
        o = self._sigmoid(gates[:, 3*H:])
        
        c_next = f * c_prev + i * g
        tanh_c_next = np.tanh(c_next)
        h_next = o * tanh_c_next
        
        self.cache = (concat, i, f, g, o, c_prev, c_next, tanh_c_next)
        return h_next, c_next
    
    def backward(self, dh_next, dc_next):
        """LSTMセルの逆伝搬
        
        Args:
            dh_next: h_tに対する上流勾配 (batch_size, hidden_dim)
            dc_next: c_tに対する上流勾配 (batch_size, hidden_dim)
        
        Returns:
            dx_t:   x_tに対する勾配 (batch_size, input_dim)
            dh_prev: h_{t-1}に対する勾配 (batch_size, hidden_dim)
            dc_prev: c_{t-1}に対する勾配 (batch_size, hidden_dim)
        """
        H = self.hidden_dim
        concat, i, f, g, o, c_prev, c_next, tanh_c_next = self.cache
        
        # --- ステップ1: h_t = o * tanh(c_t) の逆伝搬 ---
        # dh_next → do, dc_t
        do = dh_next * tanh_c_next                        # (batch, H)
        dc = dc_next + dh_next * o * (1 - tanh_c_next**2)  # (batch, H)
        
        # --- ステップ2: c_t = f * c_{t-1} + i * g の逆伝搬 ---
        df = dc * c_prev        # 忘却ゲートの勾配
        dc_prev = dc * f        # セル状態の勾配（加算的！）
        di = dc * g             # 入力ゲートの勾配
        dg = dc * i             # セル候補の勾配
        
        # --- ステップ3: 活性化関数の逆伝搬 ---
        # シグモイド: σ'(x) = σ(x)(1-σ(x))
        di_raw = di * i * (1 - i)    # 入力ゲート（シグモイド）
        df_raw = df * f * (1 - f)    # 忘却ゲート（シグモイド）
        dg_raw = dg * (1 - g**2)     # セル候補（tanh）
        do_raw = do * o * (1 - o)    # 出力ゲート（シグモイド）
        
        # ゲート勾配を結合
        dgates = np.concatenate([di_raw, df_raw, dg_raw, do_raw], axis=1)  # (batch, 4H)
        
        # --- ステップ4: 重み・バイアス・入力の勾配 ---
        self.dW += concat.T @ dgates            # (concat_dim, 4H)
        self.db += dgates.sum(axis=0)           # (4H,)
        
        dconcat = dgates @ self.W.T             # (batch, concat_dim)
        dx_t = dconcat[:, :self.input_dim]      # (batch, input_dim)
        dh_prev = dconcat[:, self.input_dim:]   # (batch, hidden_dim)
        
        return dx_t, dh_prev, dc_prev
    
    def reset_grads(self):
        """勾配をゼロにリセット"""
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)


# --- テスト ---
np.random.seed(42)
cell = LSTMCell(input_dim=3, hidden_dim=5)

x_t = np.random.randn(4, 3)
h_prev = np.random.randn(4, 5) * 0.1
c_prev = np.random.randn(4, 5) * 0.1

h_next, c_next = cell.forward(x_t, h_prev, c_prev)

dh_next = np.random.randn(4, 5)
dc_next = np.random.randn(4, 5)

dx_t, dh_prev, dc_prev = cell.backward(dh_next, dc_next)

print("=== LSTMCell Backward テスト ===")
print(f"dx_t:    shape={dx_t.shape}")
print(f"dh_prev: shape={dh_prev.shape}")
print(f"dc_prev: shape={dc_prev.shape}")
print(f"dW:      shape={cell.dW.shape}, norm={np.linalg.norm(cell.dW):.4f}")
print(f"db:      shape={cell.db.shape}, norm={np.linalg.norm(cell.db):.4f}")

In [ ]:
# ============================================================
# シーケンス全体を処理するLSTMクラス
# ============================================================

class LSTM:
    """シーケンス処理用LSTMレイヤー + 出力層"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.hidden_dim = hidden_dim
        self.cell = LSTMCell(input_dim, hidden_dim)
        
        # 出力層
        scale_out = np.sqrt(2.0 / hidden_dim)
        self.W_out = np.random.randn(hidden_dim, output_dim) * scale_out
        self.b_out = np.zeros(output_dim)
        self.dW_out = np.zeros_like(self.W_out)
        self.db_out = np.zeros_like(self.b_out)
        
        # シーケンスのキャッシュ
        self.seq_cache = []
    
    def forward(self, X):
        """シーケンスの順伝搬
        
        Args:
            X: (batch_size, seq_len, input_dim)
        
        Returns:
            y: (batch_size, output_dim)  最後の時刻の出力
        """
        batch_size, seq_len, _ = X.shape
        H = self.hidden_dim
        
        h = np.zeros((batch_size, H))
        c = np.zeros((batch_size, H))
        
        self.seq_cache = []
        self.X = X
        self.h_list = [h]
        self.c_list = [c]
        
        for t in range(seq_len):
            h, c = self.cell.forward(X[:, t, :], h, c)
            self.seq_cache.append(self.cell.cache)
            self.h_list.append(h)
            self.c_list.append(c)
        
        # 最後の隠れ状態から出力
        y = h @ self.W_out + self.b_out
        self.final_h = h
        
        return y
    
    def backward(self, dy):
        """シーケンス全体の逆伝搬 (BPTT)
        
        Args:
            dy: 出力の勾配 (batch_size, output_dim)
        """
        seq_len = len(self.seq_cache)
        H = self.hidden_dim
        
        # 出力層の勾配
        self.dW_out = self.final_h.T @ dy
        self.db_out = dy.sum(axis=0)
        dh = dy @ self.W_out.T
        dc = np.zeros_like(dh)
        
        # 勾配の初期化
        self.cell.reset_grads()
        
        # 時間方向に逆伝搬
        self.grad_norms = []  # 可視化用
        for t in reversed(range(seq_len)):
            self.cell.cache = self.seq_cache[t]
            dx_t, dh, dc = self.cell.backward(dh, dc)
            self.grad_norms.append(np.linalg.norm(dh))
        
        self.grad_norms = self.grad_norms[::-1]  # 時間順に戻す
    
    def update_params(self, lr, clip_value=5.0):
        """勾配クリッピング + SGDパラメータ更新"""
        for param, grad in [(self.cell.W, self.cell.dW),
                            (self.cell.b, self.cell.db),
                            (self.W_out, self.dW_out),
                            (self.b_out, self.db_out)]:
            np.clip(grad, -clip_value, clip_value, out=grad)
        
        self.cell.W -= lr * self.cell.dW
        self.cell.b -= lr * self.cell.db
        self.W_out -= lr * self.dW_out
        self.b_out -= lr * self.db_out


# --- テスト ---
np.random.seed(42)
model = LSTM(input_dim=2, hidden_dim=16, output_dim=1)
X_test = np.random.randn(8, 20, 2)  # batch=8, seq_len=20, input=2
y_pred = model.forward(X_test)
print(f"入力: {X_test.shape} → 出力: {y_pred.shape}")
print(f"出力値: {y_pred[:3, 0].round(4)}")

In [ ]:
# ============================================================
# Section 6: 勾配チェック（数値勾配 vs 解析的勾配）
# ============================================================

def numerical_gradient(cell, x_t, h_prev, c_prev, dh_next, dc_next, param_name, eps=1e-5):
    """数値微分による勾配計算"""
    param = getattr(cell, param_name)
    grad = np.zeros_like(param)
    
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old_val = param[idx]
        
        # f(x + eps)
        param[idx] = old_val + eps
        h_plus, c_plus = cell.forward(x_t, h_prev, c_prev)
        loss_plus = np.sum(h_plus * dh_next) + np.sum(c_plus * dc_next)
        
        # f(x - eps)
        param[idx] = old_val - eps
        h_minus, c_minus = cell.forward(x_t, h_prev, c_prev)
        loss_minus = np.sum(h_minus * dh_next) + np.sum(c_minus * dc_next)
        
        # 中心差分
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        
        param[idx] = old_val
        it.iternext()
    
    return grad


# 小さな次元でテスト
np.random.seed(42)
input_dim_gc = 2
hidden_dim_gc = 3
batch_size_gc = 2

cell_gc = LSTMCell(input_dim_gc, hidden_dim_gc)
x_gc = np.random.randn(batch_size_gc, input_dim_gc)
h_gc = np.random.randn(batch_size_gc, hidden_dim_gc) * 0.1
c_gc = np.random.randn(batch_size_gc, hidden_dim_gc) * 0.1
dh_gc = np.random.randn(batch_size_gc, hidden_dim_gc)
dc_gc = np.random.randn(batch_size_gc, hidden_dim_gc)

# 解析的勾配
cell_gc.forward(x_gc, h_gc, c_gc)
cell_gc.reset_grads()
cell_gc.backward(dh_gc, dc_gc)
analytical_dW = cell_gc.dW.copy()
analytical_db = cell_gc.db.copy()

# 数値勾配
numerical_dW = numerical_gradient(cell_gc, x_gc, h_gc, c_gc, dh_gc, dc_gc, 'W')
numerical_db = numerical_gradient(cell_gc, x_gc, h_gc, c_gc, dh_gc, dc_gc, 'b')

# 相対誤差
def relative_error(a, b):
    return np.max(np.abs(a - b) / (np.maximum(np.abs(a) + np.abs(b), 1e-8)))

err_W = relative_error(analytical_dW, numerical_dW)
err_b = relative_error(analytical_db, numerical_db)

print("=" * 60)
print("勾配チェック結果")
print("=" * 60)
print(f"dW の相対誤差: {err_W:.2e}  {'OK' if err_W < 1e-5 else 'NG'}")
print(f"db の相対誤差: {err_b:.2e}  {'OK' if err_b < 1e-5 else 'NG'}")
print()
if err_W < 1e-5 and err_b < 1e-5:
    print("全ての勾配チェックに合格しました！実装は正しいです。")
else:
    print("勾配チェックに失敗しました。実装を確認してください。")

## 7. Adding Problemでの実証

### Adding Problemとは

長距離依存性をテストする古典的なベンチマーク（Hochreiter & Schmidhuber, 1997）：

- 入力: 長さ $T$ のシーケンス、各時刻に $(\text{value}, \text{mask})$ のペア
  - `value`: $[0, 1)$ の一様乱数
  - `mask`: シーケンス中の2箇所だけ1、他は0
- 出力: `mask == 1` の位置にある `value` の合計

**ベースラインMSE**: mask位置がランダムなので、常に1.0を予測すると $\text{MSE} \approx 0.167$

RNNは $T$ が大きいとベースラインから改善できないが、LSTMは成功する。

In [ ]:
# ============================================================
# Adding Problem: データ生成
# ============================================================

def generate_adding_problem(n_samples, seq_len):
    """Adding Problemのデータ生成
    
    Args:
        n_samples: サンプル数
        seq_len: シーケンス長
    
    Returns:
        X: (n_samples, seq_len, 2)  [values, mask]
        y: (n_samples, 1)           マスクされた値の合計
    """
    X = np.zeros((n_samples, seq_len, 2))
    y = np.zeros((n_samples, 1))
    
    # ランダムな値
    X[:, :, 0] = np.random.uniform(0, 1, (n_samples, seq_len))
    
    for i in range(n_samples):
        # 2つのランダムな位置を選択
        # 1つは前半、もう1つは後半から（長距離依存性を保証）
        pos1 = np.random.randint(0, seq_len // 2)
        pos2 = np.random.randint(seq_len // 2, seq_len)
        X[i, pos1, 1] = 1.0
        X[i, pos2, 1] = 1.0
        y[i, 0] = X[i, pos1, 0] + X[i, pos2, 0]
    
    return X, y


# --- テスト ---
np.random.seed(42)
X_demo, y_demo = generate_adding_problem(5, 20)

print("Adding Problem サンプル:")
for i in range(3):
    mask_positions = np.where(X_demo[i, :, 1] == 1)[0]
    values = [X_demo[i, p, 0] for p in mask_positions]
    print(f"  サンプル{i}: マスク位置={mask_positions}, 値={[f'{v:.3f}' for v in values]}, 合計={y_demo[i,0]:.3f}")

# ベースラインMSE（常に1.0を予測）
X_baseline, y_baseline = generate_adding_problem(10000, 50)
baseline_mse = np.mean((y_baseline - 1.0)**2)
print(f"\nベースラインMSE（常に1.0を予測）: {baseline_mse:.4f}")
print(f"理論値: ~0.167")

In [ ]:
# ============================================================
# バニラRNN実装（比較用）
# ============================================================

class SimpleRNN:
    """バニラRNN（比較用）"""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.hidden_dim = hidden_dim
        
        scale_x = np.sqrt(2.0 / input_dim)
        scale_h = np.sqrt(2.0 / hidden_dim)
        self.W_xh = np.random.randn(input_dim, hidden_dim) * scale_x
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * scale_h
        self.b_h = np.zeros(hidden_dim)
        
        scale_out = np.sqrt(2.0 / hidden_dim)
        self.W_out = np.random.randn(hidden_dim, output_dim) * scale_out
        self.b_out = np.zeros(output_dim)
    
    def forward(self, X):
        batch_size, seq_len, _ = X.shape
        H = self.hidden_dim
        h = np.zeros((batch_size, H))
        
        self.X = X
        self.h_list = [h]
        
        for t in range(seq_len):
            h = np.tanh(X[:, t, :] @ self.W_xh + h @ self.W_hh + self.b_h)
            self.h_list.append(h)
        
        y = h @ self.W_out + self.b_out
        self.final_h = h
        return y
    
    def backward(self, dy):
        seq_len = self.X.shape[1]
        H = self.hidden_dim
        
        # 出力層の勾配
        self.dW_out = self.final_h.T @ dy
        self.db_out = dy.sum(axis=0)
        dh = dy @ self.W_out.T
        
        self.dW_xh = np.zeros_like(self.W_xh)
        self.dW_hh = np.zeros_like(self.W_hh)
        self.db_h = np.zeros_like(self.b_h)
        
        self.grad_norms = []
        
        for t in reversed(range(seq_len)):
            h_t = self.h_list[t + 1]
            h_prev = self.h_list[t]
            
            # tanh の導関数
            dtanh = dh * (1 - h_t**2)
            
            self.dW_xh += self.X[:, t, :].T @ dtanh
            self.dW_hh += h_prev.T @ dtanh
            self.db_h += dtanh.sum(axis=0)
            
            dh = dtanh @ self.W_hh.T
            self.grad_norms.append(np.linalg.norm(dh))
        
        self.grad_norms = self.grad_norms[::-1]
    
    def update_params(self, lr, clip_value=5.0):
        for param_name in ['W_xh', 'W_hh', 'b_h', 'W_out', 'b_out']:
            grad = getattr(self, 'd' + param_name)
            np.clip(grad, -clip_value, clip_value, out=grad)
            param = getattr(self, param_name)
            param -= lr * grad


print("SimpleRNN 準備完了")

In [ ]:
# ============================================================
# Adding Problem: RNN vs LSTM 訓練
# ============================================================

def train_model(model, X_train, y_train, X_val, y_val, 
                n_epochs=150, batch_size=32, lr=0.01):
    """モデルの訓練ループ"""
    n_train = X_train.shape[0]
    train_losses = []
    val_losses = []
    
    for epoch in range(n_epochs):
        # ミニバッチSGD
        indices = np.random.permutation(n_train)
        epoch_loss = 0.0
        n_batches = 0
        
        for start in range(0, n_train, batch_size):
            end = min(start + batch_size, n_train)
            batch_idx = indices[start:end]
            X_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]
            
            # Forward
            y_pred = model.forward(X_batch)
            
            # MSE損失
            loss = np.mean((y_pred - y_batch)**2)
            epoch_loss += loss
            n_batches += 1
            
            # Backward
            dy = 2.0 * (y_pred - y_batch) / y_batch.shape[0]
            model.backward(dy)
            
            # Update
            model.update_params(lr)
        
        # エポック損失
        train_losses.append(epoch_loss / n_batches)
        
        # 検証損失
        y_val_pred = model.forward(X_val)
        val_loss = np.mean((y_val_pred - y_val)**2)
        val_losses.append(val_loss)
        
        if (epoch + 1) % 30 == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_losses[-1]:.4f}, val_loss={val_loss:.4f}")
    
    return train_losses, val_losses


# --- 実験設定 ---
results = {}

for seq_len in [50, 100]:
    print(f"\n{'='*60}")
    print(f"シーケンス長 T = {seq_len}")
    print(f"{'='*60}")
    
    np.random.seed(42)
    X_train, y_train = generate_adding_problem(1000, seq_len)
    X_val, y_val = generate_adding_problem(200, seq_len)
    
    # RNN
    print("\n--- RNN ---")
    np.random.seed(42)
    rnn = SimpleRNN(input_dim=2, hidden_dim=32, output_dim=1)
    rnn_train, rnn_val = train_model(rnn, X_train, y_train, X_val, y_val,
                                      n_epochs=150, batch_size=32, lr=0.001)
    
    # LSTM
    print("\n--- LSTM ---")
    np.random.seed(42)
    lstm = LSTM(input_dim=2, hidden_dim=32, output_dim=1)
    lstm_train, lstm_val = train_model(lstm, X_train, y_train, X_val, y_val,
                                        n_epochs=150, batch_size=32, lr=0.01)
    
    results[seq_len] = {
        'rnn_train': rnn_train, 'rnn_val': rnn_val,
        'lstm_train': lstm_train, 'lstm_val': lstm_val,
        'rnn_model': rnn, 'lstm_model': lstm
    }

print("\n訓練完了！")

In [ ]:
# ============================================================
# Adding Problem: 結果の可視化
# ============================================================

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

seq_lens = [50, 100]
baseline = 0.167

# --- 学習曲線 ---
for idx, sl in enumerate(seq_lens):
    ax = fig.add_subplot(gs[0, idx])
    r = results[sl]
    epochs = range(1, len(r['rnn_val']) + 1)
    
    ax.plot(epochs, r['rnn_val'], 'r-', linewidth=2, label='RNN (val)', alpha=0.8)
    ax.plot(epochs, r['lstm_val'], 'b-', linewidth=2, label='LSTM (val)', alpha=0.8)
    ax.axhline(y=baseline, color='gray', linestyle='--', linewidth=1.5, 
               label=f'ベースライン ({baseline})', alpha=0.7)
    ax.set_xlabel('エポック', fontsize=12)
    ax.set_ylabel('MSE（検証）', fontsize=12)
    ax.set_title(f'学習曲線（T={sl}）', fontsize=14)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 0.25)

# --- 最終MSE棒グラフ ---
ax = fig.add_subplot(gs[1, 0])
x_pos = np.arange(len(seq_lens))
width = 0.25

rnn_finals = [results[sl]['rnn_val'][-1] for sl in seq_lens]
lstm_finals = [results[sl]['lstm_val'][-1] for sl in seq_lens]

bars1 = ax.bar(x_pos - width, rnn_finals, width, label='RNN', color='red', alpha=0.7)
bars2 = ax.bar(x_pos, lstm_finals, width, label='LSTM', color='blue', alpha=0.7)
bars3 = ax.bar(x_pos + width, [baseline]*len(seq_lens), width, label='ベースライン', color='gray', alpha=0.5)

ax.set_xticks(x_pos)
ax.set_xticklabels([f'T={sl}' for sl in seq_lens], fontsize=12)
ax.set_ylabel('最終MSE', fontsize=12)
ax.set_title('最終MSE比較', fontsize=14)
ax.legend(fontsize=10)

# 値のアノテーション
for bar_group in [bars1, bars2]:
    for bar in bar_group:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

# --- 結果サマリー ---
ax = fig.add_subplot(gs[1, 1])
ax.axis('off')

summary_text = "Adding Problem 結果サマリー\n" + "=" * 40 + "\n\n"
for sl in seq_lens:
    r = results[sl]
    rnn_mse = r['rnn_val'][-1]
    lstm_mse = r['lstm_val'][-1]
    summary_text += f"T = {sl}:\n"
    summary_text += f"  RNN:      MSE = {rnn_mse:.4f}"
    summary_text += f"  {'(失敗)' if rnn_mse > 0.1 else '(成功)'}\n"
    summary_text += f"  LSTM:     MSE = {lstm_mse:.4f}"
    summary_text += f"  {'(失敗)' if lstm_mse > 0.1 else '(成功)'}\n"
    summary_text += f"  Baseline: MSE = {baseline:.4f}\n\n"

summary_text += "RNNはベースライン付近に停滞\nLSTMは長距離依存性を学習可能"

ax.text(0.1, 0.95, summary_text, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', font,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Adding Problem: RNN vs LSTM', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. ゲート活性化の可視化

訓練済みLSTMにAdding Problemのシーケンスを入力し、各ゲートがどのように動作するかを観察します。

**注目ポイント**:
- 忘却ゲート $f_t$: マスク位置の情報を記憶するため、ほぼ1を維持するか
- 入力ゲート $i_t$: マスク位置（重要な情報）で活性化するか
- 出力ゲート $o_t$: シーケンス末尾で出力するために開くか

In [ ]:
# ============================================================
# ゲート活性化ヒートマップ
# ============================================================

# 訓練済みLSTM (T=50) でシーケンスを処理
np.random.seed(123)
lstm_model = results[50]['lstm_model']
X_vis, y_vis = generate_adding_problem(1, 50)

# forwardを実行してゲート値を記録
batch_size_vis = 1
seq_len_vis = 50
H_vis = lstm_model.hidden_dim

h = np.zeros((batch_size_vis, H_vis))
c = np.zeros((batch_size_vis, H_vis))

gate_i_history = []
gate_f_history = []
gate_o_history = []
gate_g_history = []

for t in range(seq_len_vis):
    h, c = lstm_model.cell.forward(X_vis[:, t, :], h, c)
    concat, i, f, g, o, c_prev, c_next, tanh_c = lstm_model.cell.cache
    gate_i_history.append(i[0])  # (hidden_dim,)
    gate_f_history.append(f[0])
    gate_g_history.append(g[0])
    gate_o_history.append(o[0])

gate_i_arr = np.array(gate_i_history)  # (seq_len, hidden_dim)
gate_f_arr = np.array(gate_f_history)
gate_g_arr = np.array(gate_g_history)
gate_o_arr = np.array(gate_o_history)

# マスク位置
mask_positions = np.where(X_vis[0, :, 1] == 1)[0]

# --- ヒートマップ ---
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

gate_data = [
    (gate_f_arr, '忘却ゲート $f_t$\n（1=記憶保持, 0=忘却）', 'Reds'),
    (gate_i_arr, '入力ゲート $i_t$\n（1=書き込み, 0=無視）', 'Blues'),
    (gate_g_arr, 'セル候補 $\\tilde{c}_t$\n（書き込む値）', 'RdBu_r'),
    (gate_o_arr, '出力ゲート $o_t$\n（1=出力, 0=遮断）', 'Greens'),
]

for ax, (data, title, cmap) in zip(axes, gate_data):
    # 最初の16ユニットのみ表示
    n_units = min(16, data.shape[1])
    im = ax.imshow(data[:, :n_units].T, aspect='auto', cmap=cmap,
                   interpolation='nearest')
    ax.set_ylabel(title, fontsize=10)
    ax.set_yticks(range(0, n_units, 4))
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
    
    # マスク位置を縦線で表示
    for mp in mask_positions:
        ax.axvline(x=mp, color='white', linestyle='--', linewidth=1.5, alpha=0.8)

axes[-1].set_xlabel('時間ステップ', fontsize=12)

plt.suptitle(f'LSTMゲート活性化ヒートマップ\n（白破線=マスク位置 {mask_positions}、値の合計={y_vis[0,0]:.3f}）',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"マスク位置: {mask_positions}")
print(f"忘却ゲート平均値: {gate_f_arr.mean():.3f}（高い→記憶を保持している）")
print(f"入力ゲート平均値: {gate_i_arr.mean():.3f}")

In [ ]:
# ============================================================
# RNN vs LSTM: 勾配ノルムの比較
# ============================================================

# T=50のモデルで勾配ノルムを取得
np.random.seed(42)
X_grad, y_grad = generate_adding_problem(32, 50)

# RNNの勾配ノルム
rnn_model = results[50]['rnn_model']
y_pred_rnn = rnn_model.forward(X_grad)
dy_rnn = 2.0 * (y_pred_rnn - y_grad) / y_grad.shape[0]
rnn_model.backward(dy_rnn)
rnn_gnorms = rnn_model.grad_norms

# LSTMの勾配ノルム
lstm_model_50 = results[50]['lstm_model']
y_pred_lstm = lstm_model_50.forward(X_grad)
dy_lstm = 2.0 * (y_pred_lstm - y_grad) / y_grad.shape[0]
lstm_model_50.backward(dy_lstm)
lstm_gnorms = lstm_model_50.grad_norms

# --- 可視化 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 線形スケール
ax = axes[0]
ax.plot(range(len(rnn_gnorms)), rnn_gnorms, 'r-', linewidth=2, label='RNN', alpha=0.7)
ax.plot(range(len(lstm_gnorms)), lstm_gnorms, 'b-', linewidth=2, label='LSTM', alpha=0.7)
ax.set_xlabel('時間ステップ', fontsize=12)
ax.set_ylabel('勾配ノルム', fontsize=12)
ax.set_title('勾配ノルムの推移（線形スケール）', fontsize=14)
ax.legend(fontsize=12)

# 対数スケール
ax = axes[1]
# ゼロを避ける
rnn_gnorms_safe = np.array(rnn_gnorms) + 1e-10
lstm_gnorms_safe = np.array(lstm_gnorms) + 1e-10
ax.semilogy(range(len(rnn_gnorms_safe)), rnn_gnorms_safe, 'r-', linewidth=2, label='RNN', alpha=0.7)
ax.semilogy(range(len(lstm_gnorms_safe)), lstm_gnorms_safe, 'b-', linewidth=2, label='LSTM', alpha=0.7)
ax.set_xlabel('時間ステップ', fontsize=12)
ax.set_ylabel('勾配ノルム（対数）', fontsize=12)
ax.set_title('勾配ノルムの推移（対数スケール）', fontsize=14)
ax.legend(fontsize=12)

plt.suptitle('訓練済みモデルの勾配フロー比較（T=50）',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"勾配ノルム（最初の時刻 / 最後の時刻）:")
print(f"  RNN:  {rnn_gnorms[0]:.6f} / {rnn_gnorms[-1]:.6f}  比率: {rnn_gnorms[0]/(rnn_gnorms[-1]+1e-10):.1f}x")
print(f"  LSTM: {lstm_gnorms[0]:.6f} / {lstm_gnorms[-1]:.6f}  比率: {lstm_gnorms[0]/(lstm_gnorms[-1]+1e-10):.1f}x")

## 9. まとめ

### 学んだこと

1. **RNNの限界**: 勾配消失により長距離依存性を学習できない
2. **LSTMの4ゲート**: 忘却・入力・セル候補・出力の4つの機構で情報フローを制御
3. **セル状態の高速道路**: 加算的更新 ($c_t = f_t \odot c_{t-1} + ...$) により勾配が長距離を伝搬
4. **実装**: Forward/Backwardを完全にNumPyで実装し、勾配チェックで正しさを検証
5. **Adding Problem**: RNNが失敗する長距離タスクでLSTMは成功

### チートシート: RNN vs LSTM

| 特性 | バニラRNN | LSTM |
|------|----------|------|
| パラメータ数 | $O(H^2)$ | $O(4H^2)$ ← 4倍 |
| 状態変数 | $h_t$ のみ | $h_t$ と $c_t$ |
| 勾配伝搬 | $\prod W_{hh}$（行列積） | $\prod f_t$（要素積） |
| 長期依存性 | 困難（~10-20ステップ） | 可能（~100+ステップ） |
| 情報制御 | なし | ゲートで選択的 |
| 計算コスト | 低い | RNNの約4倍 |
| 忘却ゲートバイアス | N/A | 1.0で初期化（重要） |

### よくある間違い

1. **忘却ゲートバイアスの初期化を忘れる**: $b_f = 0$ だと $f_t \approx 0.5$ で情報が急速に失われる。$b_f = 1$ で初期化すること
2. **セル状態とhidden状態を混同**: $c_t$ は内部記憶、$h_t$ は出力。次のレイヤーに渡すのは $h_t$
3. **勾配クリッピングを忘れる**: LSTMでも勾配爆発は起こりうる。クリッピングは必須
4. **ゲート行列を別々に計算**: 4つの行列を結合して1回の行列積で計算するのが効率的

### 学習チェックリスト

- [ ] LSTMの4ゲートの役割をそれぞれ説明できる
- [ ] セル状態が勾配消失を解決するメカニズムを説明できる
- [ ] LSTMCell の forward/backward を自力で実装できる
- [ ] 忘却ゲートバイアス初期化の重要性を理解している
- [ ] Adding Problem でRNNとLSTMの性能差を説明できる

### 次のステップ

→ **Notebook 124: GRU（Gated Recurrent Unit）** — LSTMを簡略化した2ゲートモデル

## 10. 自己評価クイズ

以下の問いに答えて、理解度を確認しましょう。

---

**Q1**: 忘却ゲートのバイアスを1.0で初期化するのはなぜですか？

<details><summary>回答を見る</summary>

訓練初期に $f_t = \sigma(W_f z + b_f)$ の値を1に近くするためです。$b_f = 1$ なら $\sigma(1) \approx 0.73$ となり、セル状態が保持されやすくなります。$b_f = 0$ だと $\sigma(0) = 0.5$ で情報が急速に失われ、長期依存性の学習が困難になります（Gers et al., 2000で提案された重要なトリック）。

</details>

---

**Q2**: LSTMはどのようにして勾配消失問題を解決していますか？

<details><summary>回答を見る</summary>

セル状態の更新が $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ という**加算的**構造であることがポイントです。$c_t$ から $c_{t-1}$ への勾配は $\frac{\partial c_t}{\partial c_{t-1}} = f_t$（要素ごとの積）であり、$f_t \approx 1$ なら勾配はほぼ減衰せずに長距離を伝搬できます。RNNでは $W_{hh}$ との行列積が繰り返されるため、指数的に減衰/爆発します。

</details>

---

**Q3**: すべての忘却ゲートが0（$f_t = 0$ for all $t$）だった場合、LSTMはどのように振る舞いますか？

<details><summary>回答を見る</summary>

$f_t = 0$ の場合、$c_t = 0 \cdot c_{t-1} + i_t \odot \tilde{c}_t = i_t \odot \tilde{c}_t$ となり、過去のセル状態が完全に消去されます。つまり、LSTMは**メモリのないモデル**に退化し、各時刻で新しい情報のみに依存します。これはバニラRNNよりもさらに記憶能力が低い状態です。だからこそ忘却ゲートのバイアス初期化が重要なのです。

</details>

---

**Q4**: 4つのゲート行列を1つの大きな行列に結合する利点は何ですか？

<details><summary>回答を見る</summary>

計算効率の向上です。4つの別々の行列積 $(z W_i, z W_f, z W_c, z W_o)$ を行う代わりに、1つの大きな行列積 $z W$ （$W$ は結合行列）で済みます。GPUのような並列計算ハードウェアでは、大きな行列積1回の方が小さな行列積4回よりも大幅に高速です（メモリアクセスの局所性、カーネル起動オーバーヘッドの削減）。

</details>

---

**Q5**: LSTMの出力ゲート $o_t$ が存在しないと仮定した場合（つまり $h_t = \tanh(c_t)$）、何が問題になりますか？

<details><summary>回答を見る</summary>

出力ゲートがないと、セル状態のすべての情報が常に外部に露出してしまいます。出力ゲートは「いつ・どの情報を出力するか」を制御する重要な役割を持ちます。例えばAdding Problemでは、シーケンスの途中では情報を蓄積するだけで出力を抑制し、最後の時刻でのみ出力する、という動作が必要です。出力ゲートなしではこのような選択的な情報公開ができません。

</details>

## 参考文献

1. **Hochreiter, S., & Schmidhuber, J.** (1997). *Long Short-Term Memory*. Neural Computation, 9(8), 1735-1780. — LSTMの原論文
2. **Gers, F. A., Schmidhuber, J., & Cummins, F.** (2000). *Learning to Forget: Continual Prediction with LSTM*. Neural Computation, 12(10), 2451-2471. — 忘却ゲートの提案
3. **Greff, K., Srivastava, R. K., Koutník, J., Steunebrink, B. R., & Schmidhuber, J.** (2017). *LSTM: A Search Space Odyssey*. IEEE Transactions on Neural Networks and Learning Systems. — LSTMバリアントの網羅的比較
4. **Olah, C.** (2015). *Understanding LSTM Networks*. https://colah.github.io/posts/2015-08-Understanding-LSTMs/ — 直感的な解説ブログ